💡 **Environment:** `clamp-analyses`  


# Description

Projects LINCS L1000 consensus drug signatures into the ARCHS4 CLAMP latent space.

**Input**: `data/drug_disease_associations/lincs-data.pkl` (7120 Ensembl genes × 1170 drugs).

**Outputs**:
- `01_lincs_projection_archs4/lincs/lincs-data.pkl` (genes × drugs, copy)
- `01_lincs_projection_archs4/lincs/lincs-projection.pkl` (LVs × drugs)


# Modules loading

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd

import rpy2.robjects as ro
from rpy2.robjects.packages import importr
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter

from pyprojroot import here

# Settings

In [ ]:
MODEL_KEY = 'archs4'

# Input: processed LINCS data
DATA_DIR = here('data/drug_disease_associations')
LINCS_INPUT_FILE = DATA_DIR / 'lincs-data.pkl'
display(LINCS_INPUT_FILE)
assert LINCS_INPUT_FILE.exists()

CLAMP_MODEL_FILE = here('output/01_model_building/04_archs4/06_bp_coverage_rshall/06_bp_coverage_hall_rs_100/hall_coverage_rs100_seed_1/CLAMPfull_hall.rds')
display(CLAMP_MODEL_FILE)
assert CLAMP_MODEL_FILE.exists()


PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/data/drug_disease_associations/lincs-data.pkl')

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/01_model_building/04_archs4/06_bp_coverage_rshall/06_bp_coverage_hall_rs_100/hall_coverage_rs100_seed_1/CLAMPfull_hall.rds')

In [4]:
OUTPUT_DIR = here('output/03_model_biology/00_archs4/02_drug_disease_associations/01_lincs_projection_archs4') / 'lincs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
display(OUTPUT_DIR)


PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/01_lincs_projection_archs4/lincs')

# Load LINCS data

In [5]:
lincs_data = pd.read_pickle(LINCS_INPUT_FILE)
display(lincs_data.shape)
display(lincs_data.head())
assert lincs_data.index.is_unique
assert lincs_data.columns.is_unique
assert not lincs_data.isna().any().any()

(7120, 1170)

perturbagen,DB00014,DB00091,DB00121,DB00130,DB00131,DB00132,DB00136,DB00140,DB00146,DB00150,...,DB08995,DB09002,DB09004,DB09009,DB09010,DB09015,DB09019,DB09020,DB09022,DB09023
ENSG00000196839,-1.001,-1.835,1.391,1.132,0.257,1.932,0.508,1.408,0.777,0.032,...,-1.692,-0.516,-1.435,-0.317,-0.012,0.641,-0.230,-0.518,-0.177,2.146
ENSG00000170558,1.146,-1.863,0.011,-1.020,1.143,-0.115,1.327,0.310,-1.853,0.872,...,0.354,0.498,0.268,-1.084,-0.142,-0.077,0.633,-1.807,0.032,0.135
ENSG00000117020,-0.693,1.694,-0.804,-0.164,1.145,-1.465,1.221,-0.747,0.829,-0.961,...,-1.196,-0.230,-1.049,-0.347,0.586,0.865,-0.021,2.180,-0.956,0.105
ENSG00000133997,-0.037,0.383,0.269,-0.997,0.185,-0.536,0.424,-0.119,-1.313,0.579,...,-0.343,0.116,-0.245,-0.127,-1.367,0.149,0.117,2.084,1.178,0.772
ENSG00000101473,0.162,-0.899,0.105,-0.090,-1.291,1.404,0.185,0.157,-0.327,-0.026,...,-0.136,-1.115,-0.280,0.200,0.638,-0.197,-0.360,-2.302,-0.117,-0.167


In [6]:
# Verify all index entries are Ensembl IDs (15 chars)
_tmp = pd.Series(lincs_data.index.map(len)).value_counts()
display(_tmp)
assert _tmp.shape[0] == 1

15    7120
Name: count, dtype: int64

# Save raw LINCS data to output directory

In [7]:
output_raw_file = OUTPUT_DIR / 'lincs-data.pkl'
display(output_raw_file)
lincs_data.to_pickle(output_raw_file)
print('Saved.')

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/01_lincs_projection_archs4/lincs/lincs-data.pkl')

Saved.


# Load CLAMP model and prepare gene mapping

In [8]:
CLAMP = importr('CLAMP')
readRDS = ro.r['readRDS']
clamp = readRDS(str(CLAMP_MODEL_FILE))
print('CLAMP model loaded')

CLAMP model loaded


In [9]:
gene_symbols = list(ro.r['rownames'](clamp.rx2('Z')))
lv_names = list(ro.r['colnames'](clamp.rx2('Z')))
print(f'CLAMP genes: {len(gene_symbols)}, LVs: {len(lv_names)}')

CLAMP genes: 18423, LVs: 1728


In [10]:
# Map CLAMP gene symbols (HGNC) → Ensembl IDs
clusterProfiler = importr('clusterProfiler')

bitr_result = clusterProfiler.bitr(
    ro.StrVector(gene_symbols),
    fromType='SYMBOL',
    toType='ENSEMBL',
    OrgDb='org.Hs.eg.db',
)

with localconverter(ro.default_converter + pandas2ri.converter):
    mapping_df = ro.conversion.rpy2py(bitr_result)

print(f'Raw mapping shape: {mapping_df.shape}')
display(mapping_df.head())

R callback write-console: 
  


R callback write-console: 'select()' returned 1:many mapping between keys and columns
  


Raw mapping shape: (19313, 2)


,SYMBOL,ENSEMBL
1,A1BG,ENSG00000121410
2,A1BG-AS1,ENSG00000268895
3,A2M,ENSG00000175899
4,A2M-AS1,ENSG00000245105
5,A2ML1,ENSG00000166535


In [11]:
# Keep only 1:1 unambiguous symbol ↔ Ensembl mappings
dup_symbols = mapping_df['SYMBOL'].duplicated(keep=False)
dup_ensembl = mapping_df['ENSEMBL'].duplicated(keep=False)
mapping_1to1 = mapping_df[~dup_symbols & ~dup_ensembl].set_index('SYMBOL')
print(f'1:1 mappings: {mapping_1to1.shape[0]} / {len(gene_symbols)} CLAMP genes')

mapped_symbols = mapping_1to1.index.tolist()
mapped_ensembl = mapping_1to1['ENSEMBL'].tolist()

1:1 mappings: 16038 / 18423 CLAMP genes


In [ ]:
# Build CLAMP sub-object with Z restricted to 1:1-mapped genes.
subset_Z = ro.r('function(clamp, genes) { clamp$Z <- as.matrix(clamp$Z[genes, ]); clamp }')
clamp_sub = subset_Z(clamp, ro.StrVector(mapped_symbols))
print(f'Subsetted CLAMP Z: {len(mapped_symbols)} genes x {len(lv_names)} LVs')

R callback write-console: In addition:   


R callback write-console: Warning message:
  


R callback write-console: In (function (geneID, fromType, toType, OrgDb, drop = TRUE)  :  


R callback write-console: 
   


R callback write-console:  6.63% of input gene IDs are fail to map...
  


Subsetted CLAMP Z: 16038 genes x 1728 LVs


# Project LINCS into CLAMP

In [13]:
# Align LINCS to CLAMP gene order (mapped Ensembl IDs), fill missing genes with 0
aligned = lincs_data.reindex(mapped_ensembl).fillna(0.0).values  # (n_genes, n_drugs)

r_mat = ro.r['matrix'](
    ro.FloatVector(aligned.flatten('F')),
    nrow=aligned.shape[0],
    ncol=aligned.shape[1],
)

proj_r = CLAMP.projectCLAMP(clamp_sub, newdata=r_mat)

with localconverter(ro.default_converter + pandas2ri.converter):
    proj_values = ro.conversion.rpy2py(proj_r)

lincs_projection = pd.DataFrame(proj_values, index=lv_names, columns=lincs_data.columns)
print(f'LINCS projection shape: {lincs_projection.shape}')
display(lincs_projection.head())

LINCS projection shape: (1728, 1170)


perturbagen,DB00014,DB00091,DB00121,DB00130,DB00131,DB00132,DB00136,DB00140,DB00146,DB00150,...,DB08995,DB09002,DB09004,DB09009,DB09010,DB09015,DB09019,DB09020,DB09022,DB09023
LV1,0.003127,-0.014655,0.006365,-0.054058,0.059773,-0.023714,-0.049853,-0.023722,-0.001287,-0.015632,...,-0.079593,-0.059146,0.000345,0.034516,-0.000666,0.005783,0.029754,-0.172810,0.020110,0.000471
LV2,0.018831,0.045189,0.000370,-0.011590,0.000117,0.001920,-0.003038,-0.008572,-0.006517,0.002992,...,0.011073,-0.004997,0.017900,0.001621,0.014665,-0.001264,-0.008870,-0.049142,-0.007135,0.004529
LV3,-0.005911,-0.028553,0.005108,-0.000609,0.001949,0.004189,0.008837,0.005817,0.004727,0.002902,...,-0.003031,-0.000407,-0.007506,-0.013602,-0.006100,-0.009235,-0.000900,0.004442,0.005198,-0.002058
LV4,0.022497,-0.185666,0.002055,0.000259,0.027752,0.042420,-0.016758,0.029296,-0.013967,-0.021628,...,0.010162,-0.049056,0.036576,0.012428,-0.010925,-0.005625,-0.027448,-0.277678,0.008541,-0.012307
LV5,-0.001825,0.005136,0.002183,-0.006718,0.012377,-0.007434,0.019347,0.007425,-0.003604,0.002773,...,-0.001932,-0.003488,0.000656,0.008316,-0.002905,0.007354,-0.004349,-0.034426,0.000239,-0.000238


In [14]:
assert not lincs_projection.isna().any().any()

# Save

In [15]:
output_proj_file = OUTPUT_DIR / 'lincs-projection.pkl'
display(output_proj_file)
lincs_projection.to_pickle(output_proj_file)
print('Saved.')

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/01_lincs_projection_archs4/lincs/lincs-projection.pkl')

Saved.
